# Hugging Face Tokenizers: Theory and Practical Usage

There are **two different layers** in the Hugging Face ecosystem that both get called "the tokenizer," and conflating them is a common source of confusion:

1. **`tokenizers`** — the low-level Rust library. Everything in Notebooks 01-02 used this directly: `Tokenizer`, `models.BPE`, `trainers.BpeTrainer`. It knows how to encode/decode and train a vocabulary. It knows nothing about *models* — no concept of "this is the BOS token for a causal LM," no padding-for-batching, no `AutoModel` integration.
2. **`transformers`' tokenizer classes** (`PreTrainedTokenizerFast`, `AutoTokenizer`) — a thin wrapper *around* a `tokenizers`-library `Tokenizer`, adding exactly the model-facing metadata layer 1 doesn't have: which token is BOS/EOS/PAD/UNK, padding/truncation behavior, batch encoding with attention masks, and the `save_pretrained`/`from_pretrained`/Hub-upload conventions the rest of the `transformers` ecosystem expects.

This notebook is entirely about layer 2 — what it adds on top of layer 1, and why each addition exists. You already know layer 1 cold from the first two notebooks.

## Exercise 1 — Load a real public tokenizer, verify it against what you already trust

`AutoTokenizer.from_pretrained("gpt2")` downloads and loads the *exact same* GPT-2 vocabulary `custom-gpt-50m` already uses via `tiktoken` in Notebook 01 — same vocabulary, two completely different libraries loading it.

**Predict first**: will `AutoTokenizer`'s ids for a test sentence match `tiktoken`'s exactly, token-for-token? They should, if both are really loading the same underlying vocabulary — this is a genuine cross-library correctness check, not a rhetorical question.

In [ ]:
import tiktoken
from transformers import AutoTokenizer

hf_gpt2 = AutoTokenizer.from_pretrained("gpt2")
tk_gpt2 = tiktoken.get_encoding("gpt2")

text = "The little rabbit hopped through the forest."
hf_ids = hf_gpt2.encode(text)
tk_ids = tk_gpt2.encode(text)

print("transformers ids:", hf_ids)
print("tiktoken ids:    ", tk_ids)
print("identical:", hf_ids == tk_ids)

print("\ntype(hf_gpt2):", type(hf_gpt2).__name__)
print("special tokens:", hf_gpt2.special_tokens_map)

## Exercise 2 — Wrap YOUR OWN custom tokenizer.json for `transformers`

This is the exact pattern `custom-gpt-350m/src/gpt/cli/export_vllm.py` uses for real, to make that project's checkpoint loadable via plain `AutoModelForCausalLM` (see `docs/llm-engineering/23_the_serving_engine_ecosystem_vllm_and_friends.md` for the full story). A raw `tokenizers`-library `tokenizer.json` — like `custom-gpt-350m`'s own, or the one you trained in Notebook 02 — already has the vocabulary and merges. What it does **not** have is any concept of "which token is BOS/EOS for a causal LM" — that's `transformers`-specific metadata, not something a bare `Tokenizer` object stores.

**Predict first**: `custom-gpt-350m`'s tokenizer has exactly one special token, `<|endoftext|>`, doing double duty as both document-boundary marker and (as you'll declare below) BOS/EOS. What do you think happens if you *don't* pass `bos_token`/`eos_token` at all — does wrapping still succeed, just silently missing that metadata, or does it fail outright?

In [ ]:
from pathlib import Path
from transformers import PreTrainedTokenizerFast

REPO = Path("../..").resolve()
tokenizer_json_path = REPO / "from_scratch/custom-gpt-350m/tokenizer/tokenizer.json"

# Without bos/eos declared — check what special_tokens_map looks like
wrapped_bare = PreTrainedTokenizerFast(tokenizer_file=str(tokenizer_json_path))
print("bare wrap special tokens:", wrapped_bare.special_tokens_map)

# With bos/eos declared — the actual export_vllm.py pattern
wrapped = PreTrainedTokenizerFast(
    tokenizer_file=str(tokenizer_json_path),
    bos_token="<|endoftext|>",
    eos_token="<|endoftext|>",
)
print("declared wrap special tokens:", wrapped.special_tokens_map)
print("bos_token_id:", wrapped.bos_token_id, " eos_token_id:", wrapped.eos_token_id)

# Sanity check: encoding still works identically to the raw tokenizers-library object
text = "The little rabbit hopped through the forest."
print("\nencoded:", wrapped.encode(text, add_special_tokens=False))

## Exercise 3 — Padding and `attention_mask`: the thing raw `tokenizers` doesn't hand you

Training or serving processes many sequences in one batch — a fixed-shape tensor. Real prompts have different lengths, so shorter ones need padding to match the longest one in the batch. This isn't a `tokenizers`-library concern (encode one string, get one list of ids) — it's exactly the batching-for-a-model concern `PreTrainedTokenizerFast` layers on top.

**Predict first**: encode a batch of three very different-length sentences with `padding=True`. What shape will the resulting tensor be? What does `attention_mask` need to communicate to the model about the padding — and what would happen (concretely, to attention scores) if the model attended to padding tokens as if they were real content?

In [ ]:
# Real, well-known gotcha first: GPT-2's tokenizer has no pad_token at all (it was never
# trained for batched inference) — padding=True fails outright until you set one, and the
# standard fix is reusing eos_token rather than inventing a new one:
print("pad_token before fix:", hf_gpt2.pad_token)
hf_gpt2.pad_token = hf_gpt2.eos_token
print("pad_token after fix: ", hf_gpt2.pad_token)

batch = [
    "The rabbit ran.",
    "The little bird flew high over the tall forest trees.",
    "A cat sat.",
]
# return_tensors="np" avoids pulling in torch just for this notebook — numpy is already
# a transitive dependency of transformers/tokenizers.
encoded = hf_gpt2(batch, padding=True, return_tensors="np")

print("\ninput_ids shape:", encoded["input_ids"].shape)
print("input_ids:\n", encoded["input_ids"])
print("\nattention_mask:\n", encoded["attention_mask"])

# TODO: for the shortest sentence's row, count how many 0s are in its attention_mask —
# does that match exactly how many padding positions its input_ids row has?

## Wrap-up

You've now seen the full stack, bottom to top: raw BPE mechanics (Notebook 01), training your own vocabulary (Notebook 02), and the `transformers`-specific metadata layer that turns a bare `tokenizers.Tokenizer` into something `AutoModelForCausalLM`, `Trainer`, and the Hub ecosystem all expect (this notebook). That's the *entire* tokenizer half of `docs/llm-engineering/23_the_serving_engine_ecosystem_vllm_and_friends.md`'s conversion checklist — the other half (architecture-equivalence weight mapping) is model-specific, not tokenizer-specific, and that doc covers it in depth.

Write up whichever exercise surprised you most in `../docs/`, using `TEMPLATE.md`'s shape — Exercise 3's GPT-2-has-no-pad-token gotcha is a good candidate if you hadn't hit it before, since it's one of the most commonly-rediscovered surprises in the ecosystem.